# Performance Analytics & Fund Scorecard

Comprehensive analysis of mutual fund performance metrics: returns, risk-adjusted measures (Sharpe, Sortino), alpha/beta analysis, and composite fund scoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load data
base_path = r"C:\Users\pushk\OneDrive\Desktop\AIML\Blue Stocks\mutual-fund-analytics\data\processed"
nav_df = pd.read_csv(f"{base_path}\nav_history_clean.csv")
scheme_perf_df = pd.read_csv(f"{base_path}\scheme_performance_clean.csv")

nav_df['date'] = pd.to_datetime(nav_df['date'])

print("✓ Data loaded")
print(f"  Schemes: {nav_df['amfi_code'].nunique()}")
print(f"  Date range: {nav_df['date'].min().date()} to {nav_df['date'].max().date()}")

## 1. Daily Returns Calculation

In [ ]:
# Calculate daily returns
nav_pivot = nav_df.pivot_table(index='date', columns='amfi_code', values='nav')
daily_returns = nav_pivot.pct_change().dropna()

print(f"\nDaily Returns Analysis:")
print(f"  Trading days: {len(daily_returns)}")
print(f"  Mean daily return: {daily_returns.mean().mean():.6f}")
print(f"  Mean daily volatility: {daily_returns.std().mean():.6f}")

## 2. CAGR Calculation (Compound Annual Growth Rate)

In [ ]:
def calculate_cagr(start_price, end_price, years):
    if start_price <= 0 or years <= 0:
        return np.nan
    return (np.power(end_price / start_price, 1 / years) - 1) * 100

# Calculate CAGR
cagr_results = []

for amfi in nav_df['amfi_code'].unique():
    scheme_data = nav_df[nav_df['amfi_code'] == amfi].sort_values('date')
    if len(scheme_data) < 2:
        continue
    
    latest_date = scheme_data['date'].max()
    earliest_date = scheme_data['date'].min()
    latest_nav = scheme_data[scheme_data['date'] == latest_date]['nav'].values[0]
    earliest_nav = scheme_data[scheme_data['date'] == earliest_date]['nav'].values[0]
    
    years_total = (latest_date - earliest_date).days / 365.25
    
    cagr_data = {
        'amfi_code': amfi,
        'scheme_name': scheme_data['scheme_name'].iloc[0] if 'scheme_name' in scheme_data.columns else amfi,
        'cagr_total': calculate_cagr(earliest_nav, latest_nav, years_total) if years_total >= 1 else np.nan
    }
    
    # 3-year CAGR
    date_3y = latest_date - pd.DateOffset(years=3)
    data_3y = scheme_data[scheme_data['date'] >= date_3y]
    if len(data_3y) > 0:
        nav_3y_start = data_3y['nav'].iloc[0]
        years_3y = (latest_date - data_3y['date'].iloc[0]).days / 365.25
        if years_3y >= 0.5:
            cagr_data['cagr_3y'] = calculate_cagr(nav_3y_start, latest_nav, years_3y)
    
    cagr_results.append(cagr_data)

cagr_df = pd.DataFrame(cagr_results)
print("\nTop 10 schemes by 3-year CAGR:")
print(cagr_df.nlargest(10, 'cagr_3y')[['scheme_name', 'cagr_3y', 'cagr_total']])

## 3. Risk-Adjusted Returns: Sharpe & Sortino Ratios

Sharpe Ratio = (Annual Return - Risk-Free Rate) / Volatility
- Risk-free rate: 6.5% (RBI repo rate)
- Annualization factor: √252

Sortino Ratio = Similar but uses only downside volatility (negative returns)

In [ ]:
def calculate_sharpe_ratio(returns_series, risk_free_rate=0.065):
    excess_return = returns_series.mean() * 252 - risk_free_rate
    volatility = returns_series.std() * np.sqrt(252)
    return excess_return / volatility if volatility > 0 else np.nan

def calculate_sortino_ratio(returns_series, risk_free_rate=0.065):
    excess_return = returns_series.mean() * 252 - risk_free_rate
    downside_returns = returns_series[returns_series < 0]
    downside_vol = downside_returns.std() * np.sqrt(252)
    return excess_return / downside_vol if downside_vol > 0 else np.nan

def calculate_max_drawdown(returns_series):
    cumulative = (1 + returns_series).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min() * 100

# Calculate metrics for all schemes
perf_metrics = []

for col in daily_returns.columns:
    returns = daily_returns[col].dropna()
    if len(returns) < 252:
        continue
    
    metrics = {
        'amfi_code': col,
        'annual_return': returns.mean() * 252 * 100,
        'annual_volatility': returns.std() * np.sqrt(252) * 100,
        'sharpe_ratio': calculate_sharpe_ratio(returns),
        'sortino_ratio': calculate_sortino_ratio(returns),
        'max_drawdown_pct': calculate_max_drawdown(returns)
    }
    
    scheme_info = cagr_df[cagr_df['amfi_code'] == col]
    if len(scheme_info) > 0:
        metrics['scheme_name'] = scheme_info.iloc[0]['scheme_name']
        metrics['cagr_3y'] = scheme_info.iloc[0].get('cagr_3y', np.nan)
    
    perf_metrics.append(metrics)

perf_df = pd.DataFrame(perf_metrics)

print("\nTop 10 by Sharpe Ratio:")
print(perf_df.nlargest(10, 'sharpe_ratio')[['scheme_name', 'annual_return', 'sharpe_ratio', 'max_drawdown_pct']])

## 4. Alpha & Beta Analysis

In [ ]:
# Create benchmark returns (using synthetic benchmark for this analysis)
# In production, use actual Nifty 100 returns
benchmark_returns = pd.Series(
    np.random.normal(0.00048, 0.012, len(daily_returns)),
    index=daily_returns.index
)

alpha_beta_list = []

for col in daily_returns.columns:
    fund_returns = daily_returns[col].dropna()
    bench_returns = benchmark_returns.reindex(fund_returns.index).dropna()
    
    if len(bench_returns) < 252:
        continue
    
    # OLS regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(bench_returns, fund_returns)
    
    alpha_beta_list.append({
        'amfi_code': col,
        'alpha_annual_pct': intercept * 252 * 100,
        'beta': slope,
        'r_squared': r_value ** 2,
        'tracking_error_pct': (fund_returns - bench_returns).std() * np.sqrt(252) * 100
    })

alpha_beta_df = pd.DataFrame(alpha_beta_list)

# Merge with performance metrics
perf_complete = perf_df.merge(alpha_beta_df, on='amfi_code', how='left')

print("\nTop 10 by Alpha:")
print(perf_complete.nlargest(10, 'alpha_annual_pct')[['scheme_name', 'alpha_annual_pct', 'beta', 'tracking_error_pct']])

## 5. Fund Scorecard (Composite Score 0-100)

In [ ]:
# Composite Fund Score
# Weights: 30% Return + 25% Sharpe + 20% Alpha + 15% Expense Ratio + 10% Max Drawdown

scorecard = perf_complete.copy()

# Add expense ratios (default range if not available in data)
if 'expense_ratio' in scheme_perf_df.columns:
    expense_map = dict(zip(scheme_perf_df['amfi_code'], scheme_perf_df['expense_ratio']))
    scorecard['expense_ratio'] = scorecard['amfi_code'].map(expense_map).fillna(0.8)
else:
    scorecard['expense_ratio'] = np.random.uniform(0.2, 1.5, len(scorecard))

# Calculate component scores (0-100)
scorecard['return_score'] = scorecard['cagr_3y'].rank(pct=True) * 100
scorecard['sharpe_score'] = scorecard['sharpe_ratio'].rank(pct=True) * 100
scorecard['alpha_score'] = scorecard['alpha_annual_pct'].rank(pct=True) * 100
scorecard['expense_score'] = (1 - scorecard['expense_ratio'].rank(pct=True)) * 100
scorecard['drawdown_score'] = (1 - scorecard['max_drawdown_pct'].rank(pct=True)) * 100

# Composite fund score
scorecard['fund_score'] = (
    scorecard['return_score'] * 0.30 +
    scorecard['sharpe_score'] * 0.25 +
    scorecard['alpha_score'] * 0.20 +
    scorecard['expense_score'] * 0.15 +
    scorecard['drawdown_score'] * 0.10
)

# Final scorecard output
final_scorecard = scorecard[[
    'scheme_name', 'cagr_3y', 'sharpe_ratio', 'sortino_ratio',
    'alpha_annual_pct', 'beta', 'max_drawdown_pct', 'expense_ratio', 'fund_score'
]].sort_values('fund_score', ascending=False).reset_index(drop=True)

final_scorecard['rank'] = range(1, len(final_scorecard) + 1)

print("\n" + "="*110)
print("TOP 15 FUNDS BY COMPOSITE SCORE")
print("="*110)
print(final_scorecard.head(15)[['rank', 'scheme_name', 'fund_score', 'cagr_3y', 'sharpe_ratio', 'alpha_annual_pct']].to_string(index=False))

# Save outputs
report_path = r"C:\Users\pushk\OneDrive\Desktop\AIML\Blue Stocks\mutual-fund-analytics\reports"
final_scorecard.to_csv(f"{report_path}\fund_scorecard.csv", index=False)
perf_complete.to_csv(f"{report_path}\performance_metrics.csv", index=False)
alpha_beta_df.to_csv(f"{report_path}\alpha_beta_analysis.csv", index=False)

print(f"\n✓ Scorecard and metrics saved to CSV files")

## 6. Performance Visualizations

In [ ]:
# Risk vs Return Scatter
fig1 = px.scatter(
    perf_complete.dropna(subset=['annual_volatility', 'annual_return']).head(40),
    x='annual_volatility',
    y='annual_return',
    size='sharpe_ratio',
    color='sharpe_ratio',
    hover_name='scheme_name',
    title='Risk vs Return Profile (Top 40 Schemes)',
    labels={'annual_volatility': 'Annual Volatility (%)', 'annual_return': 'Annual Return (%)'},
    color_continuous_scale='Viridis',
    height=600
)
fig1.show()
fig1.write_html(f"{report_path}\performance_risk_return.html")

# Top 15 Fund Scores
top_15 = final_scorecard.head(15).copy()
fig2 = px.bar(
    top_15,
    x='fund_score',
    y='scheme_name',
    color='fund_score',
    orientation='h',
    title='Top 15 Funds by Composite Score',
    color_continuous_scale='Greens',
    height=600
)
fig2.update_layout(hovermode='y unified', yaxis={'categoryorder': 'total ascending'})
fig2.show()
fig2.write_html(f"{report_path}\top_15_fund_scores.html")

print("✓ Charts generated and saved!")